# 11 Deep Learning — Reference Solutions

Complete solutions for the PyTorch deep learning exercises on the Legionnaires' disease cluster at Pine and Cypress Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (prevents Chinese labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]
input_dim = X_df.shape[1]


def train_model(model, y_col, max_epochs=300, patience=15, lr=1e-3):
    """Train + early stopping; return train/val losses and the best model."""
    y_np_local = df[y_col].values.astype(np.float32)
    y_tr = torch.tensor(y_np_local[train_idx]).unsqueeze(1)
    y_va = torch.tensor(y_np_local[val_idx]).unsqueeze(1)
    X_tr = torch.tensor(X_np[train_idx])
    X_va = torch.tensor(X_np[val_idx])

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    best_val_loss, counter, best_epoch = float("inf"), 0, 0
    best_state = None

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(X_va), y_va).item()
        val_losses.append(vl)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        auc_tr = roc_auc_score(y_tr.numpy(), torch.sigmoid(model(X_tr)).numpy())
        auc_va = roc_auc_score(y_va.numpy(), torch.sigmoid(model(X_va)).numpy())

    return train_losses, val_losses, best_epoch, auc_tr, auc_va

## Question 1: Change the architecture (three hidden layers)

In [ ]:
torch.manual_seed(42)

# Original architecture
model_2layer = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
n2 = sum(p.numel() for p in model_2layer.parameters())
tl2, vl2, be2, auc_tr2, auc_va2 = train_model(model_2layer, "infected")

torch.manual_seed(42)

# Three hidden layers
model_3layer = nn.Sequential(
    nn.Linear(input_dim, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
n3 = sum(p.numel() for p in model_3layer.parameters())
tl3, vl3, be3, auc_tr3, auc_va3 = train_model(model_3layer, "infected")

print("=== Architecture comparison ===")
print(f"2 hidden layers: params={n2:,}, Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}, gap={auc_tr2-auc_va2:.3f}")
print(f"3 hidden layers: params={n3:,}, Train AUC={auc_tr3:.3f}, Val AUC={auc_va3:.3f}, gap={auc_tr3-auc_va3:.3f}")
print(f"\n→ The more complex architecture has more parameters ({n3} vs {n2}), but Val AUC isn't necessarily better")
print("→ A larger Train-Val gap = more severe overfitting")

## Question 2: Task B — Predict severe cases

In [ ]:
torch.manual_seed(42)

model_b = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)

tl_b, vl_b, be_b, auc_tr_b, auc_va_b = train_model(model_b, "severe_outcome")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tl_b, label="Train Loss", color="#2c7fb8")
ax.plot(vl_b, label="Val Loss", color="#e34a33")
ax.axvline(x=be_b, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({be_b})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Learning Curve — Task B (severe_outcome)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Task B: Train AUC={auc_tr_b:.3f}, Val AUC={auc_va_b:.3f}")
print(f"Task A: Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}")
print(f"\n→ Task B (severe cases) has a lower positive rate (~24%), making prediction harder")
print("→ With a small sample and few positives, DL usually performs poorly")

## Question 3 (challenge): Add Dropout

In [ ]:
torch.manual_seed(42)

# With Dropout
model_drop = nn.Sequential(
    nn.Linear(input_dim, 32), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(16, 1),
)

tl_d, vl_d, be_d, auc_tr_d, auc_va_d = train_model(model_drop, "infected")

# Learning-curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tl2, label="Train", color="#2c7fb8")
axes[0].plot(vl2, label="Val", color="#e34a33")
axes[0].set_title("No Dropout")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(tl_d, label="Train", color="#2c7fb8")
axes[1].plot(vl_d, label="Val", color="#e34a33")
axes[1].set_title("With Dropout(0.3)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

print("=== Dropout comparison ===")
print(f"No Dropout:   Train AUC={auc_tr2:.3f}, Val AUC={auc_va2:.3f}, gap={auc_tr2-auc_va2:.3f}")
print(f"Dropout(0.3): Train AUC={auc_tr_d:.3f}, Val AUC={auc_va_d:.3f}, gap={auc_tr_d-auc_va_d:.3f}")
print(f"\n→ Dropout raises train loss (because it randomly switches off neurons)")
print("→ But the Train-Val gap is usually smaller = less overfitting")
print("→ On 280 rows, Dropout has limited effect; the real fix is more data")

### Interpretation

- **Architecture complexity**: more parameters ≠ better performance. 280 rows only need the simplest architecture
- **Task A vs B**: the positive rate affects model performance; fewer positives call for an appropriate loss or sampling strategy
- **Dropout**: the most common regularization technique in DL, but its effect is limited on an extremely small sample
- **The fundamental problem**: DL isn't reasonable for 280 rows of tabular data. DL's strengths lie in unstructured data such as images, text, and sequences
- **Practical advice**: epidemiological data usually has n < 10,000 → use sklearn; n > 100,000 with unstructured features → consider DL

## Question 4 Solution

In [ ]:
# COVID-19: small tabular-data MLP for binary classification (severe cases)
rng = np.random.default_rng(1104)
n = 900
age = rng.integers(20, 90, n); male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n); vaccinated = rng.binomial(1, 0.6, n)
logit = -6 + 0.06*age + 0.4*male + 0.7*diabetes - 1.2*vaccinated
severe = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, male, diabetes, vaccinated].astype(float); y = severe.astype(float)
print(f"COVID: n={n}, severe rate={y.mean():.1%}, features={X.shape[1]}")
from sklearn.model_selection import train_test_split


def _fit(X, y, hidden=(16,), epochs=150, lr=1e-2, seed=0, test_size=0.3):
    """Standardize + split + train a small MLP; return train/val AUC and loss curves."""
    Xs = StandardScaler().fit_transform(X)
    Xtr, Xva, ytr, yva = train_test_split(Xs, y, test_size=test_size, stratify=y, random_state=42)
    torch.manual_seed(seed)
    layers, d = [], Xtr.shape[1]
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU()]; d = h
    layers += [nn.Linear(d, 1)]
    net = nn.Sequential(*layers)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    Xv = torch.tensor(Xva, dtype=torch.float32); yv = torch.tensor(yva, dtype=torch.float32).unsqueeze(1)
    tr_l, va_l = [], []
    for _ in range(epochs):
        net.train(); opt.zero_grad()
        loss = lossf(net(Xt), yt); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            tr_l.append(loss.item()); va_l.append(lossf(net(Xv), yv).item())
    with torch.no_grad():
        ptr = torch.sigmoid(net(Xt)).numpy().ravel(); pva = torch.sigmoid(net(Xv)).numpy().ravel()
    return roc_auc_score(ytr, ptr), roc_auc_score(yva, pva), tr_l, va_l


tr, va, _, _ = _fit(X, y, hidden=(16, 8), epochs=150, seed=0)
print(f"MLP  Train AUC={tr:.3f}  Val AUC={va:.3f}")
print("Interpretation: a small MLP works on small tabular data, but it usually has no clear advantage over logistic regression,")
print("and it's harder to tune / prone to overfitting -- linear or tree-based models are often preferred for this kind of problem.")

## Question 5 Solution

In [ ]:
# Dengue: small MLP for severe DHF
rng = np.random.default_rng(1105)
n = 800
age = rng.integers(1, 80, n); secondary = rng.binomial(1, 0.45, n)
platelet = rng.normal(180, 60, n).clip(20, 400); days = rng.integers(1, 8, n)
logit = -2.5 + 1.6*secondary - 0.012*platelet + 0.15*days
dhf = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, secondary, platelet, days].astype(float); y = dhf.astype(float)
print(f"Dengue: n={n}, DHF rate={y.mean():.1%}")
tr, va, _, _ = _fit(X, y, hidden=(16,), epochs=150, seed=1)
print(f"MLP  Train AUC={tr:.3f}  Val AUC={va:.3f}")
print("Interpretation: Val AUC is close to the Chapter 10 random forest; secondary infection (ADE) is the main signal.")

## Question 6 Solution

In [ ]:
# Influenza: small NN for hospitalization prediction
rng = np.random.default_rng(1106)
n = 850
age = rng.integers(0, 95, n); chronic = rng.binomial(1, 0.25, n)
vacc = rng.binomial(1, 0.5, n); onset = rng.integers(0, 6, n)
logit = -3.5 + 0.05*age + 1.0*chronic - 0.8*vacc + 0.25*onset
hosp = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, chronic, vacc, onset].astype(float); y = hosp.astype(float)
print(f"Flu: n={n}, hospitalization rate={y.mean():.1%}")
tr, va, _, _ = _fit(X, y, hidden=(16,), epochs=150, seed=2)
print(f"MLP  Train AUC={tr:.3f}  Val AUC={va:.3f}")

## Question 7 Solution

In [ ]:
# Tuberculosis: small NN for treatment outcome prognosis
rng = np.random.default_rng(1107)
n = 800
age = rng.integers(18, 85, n); mdr = rng.binomial(1, 0.15, n)
hiv = rng.binomial(1, 0.1, n); adher = rng.uniform(0.4, 1.0, n)
logit = 2.0 - 1.8*mdr - 1.2*hiv + 3.0*(adher-0.7)
success = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, mdr, hiv, adher].astype(float); y = success.astype(float)
print(f"TB: n={n}, treatment success rate={y.mean():.1%}")
tr, va, _, _ = _fit(X, y, hidden=(16,), epochs=150, seed=3)
print(f"MLP  Train AUC={tr:.3f}  Val AUC={va:.3f}")
print("Interpretation: treatment adherence is the main positive factor for treatment success.")

## Question 8 Solution

In [ ]:
# Small-sample overfitting demo: very small n, many noise features (challenge)
rng = np.random.default_rng(1108)
n = 90                      # Very small sample
signal = rng.normal(0, 1, n)
noise = rng.normal(0, 1, (n, 30))   # 30 pure noise features
y = (1/(1+np.exp(-(1.5*signal))) > rng.uniform(0, 1, n)).astype(float)
X = np.c_[signal, noise].astype(float)   # 1 signal feature + 30 noise features
print(f"n={n}, features={X.shape[1]} (only 1 carries signal), positive rate={y.mean():.1%}")
tr, va, tl, vl = _fit(X, y, hidden=(128, 128), epochs=400, lr=1e-2, seed=4, test_size=0.4)
print(f"Oversized network: Train AUC={tr:.3f}  Val AUC={va:.3f}  → clear overfitting (train much higher than val)")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tl, label="train loss", color="#6A9BCC")
ax.plot(vl, label="validation loss", color="#D94452")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss")
ax.set_title("Small sample + large network: overfitting demo")
ax.legend(); plt.tight_layout(); plt.show()
best = int(np.argmin(vl))
print(f"validation loss is lowest at epoch {best}, then rises again → early stopping should occur here")
print("Interpretation: when the parameter count far exceeds the sample size, the network 'memorizes' training-set noise instead of learning the pattern;")
print("for small epidemiological datasets, prefer simple models, regularization, early stopping, and cross-validation -- DL may not be appropriate.")